# Phase 1 - RAG fundamentals and a reliable notebook environment

**Project:** page-cited question answering over PDF documents.  
**Learning sequence:** Understand → Design → Implement → Execute → Experiment → Evaluate → Improve.  
**Phase boundary:** this lesson builds the learning environment and inspects manually written examples. PDF parsing, chunking, embeddings, search, and LLM calls belong to later phases.

Read the Markdown cells as well as the code. Choose **Python (RAG learning)**, then use **Kernel → Restart Kernel and Run All Cells**. Saved outputs show a previous execution; running the notebook yourself checks your current environment.

## 1. Learning objectives

By the end, you should be able to:

1. Explain the problem RAG addresses and why a fluent answer can still be wrong.
2. Distinguish each stage in a document question-answering pipeline.
3. Explain how document provenance makes citations traceable.
4. Distinguish page indexes, physical PDF page numbers, and printed labels.
5. Verify the Python interpreter, virtual environment, dependencies, and notebook execution.
6. Explain what today's checks prove and what still needs evaluation.

**What we implement:** environment checks, JSON inspection, and small Python demonstrations. We do not implement any future pipeline components or call a model.

## 2. What problem does RAG solve?

A language model generates tokens using learned parameters and the input supplied to it. Parameters can encode patterns and factual associations from training, but they are not a reliable, searchable copy of your private documents. A model cannot inspect a local PDF simply because you mention its filename. An application must provide its content through a prompt or a tool.

For example, ask: **"What is the annual learning budget in our company handbook?"** A model without the handbook may produce a plausible policy based on patterns from other text. It has no dependable basis for our particular budget or its page number. Fluency measures how convincing the wording sounds, not whether a source supports it.

**Retrieval-Augmented Generation (RAG)** adds an evidence selection step before generation. The application finds relevant external passages and places them alongside the question in the model's input. Think of open-book question answering: access to a library helps only if you find the right passage, read it correctly, and cite it accurately.

Internally, many implementations can be viewed as:

```text
LLM-only: answer = model(instructions, question)
RAG:     evidence = retrieve(question, indexed documents)
         answer = model(instructions, question, evidence)
```

This is a conceptual description, not runnable code. Retrieval does not usually update the model's weights. Updating the external document collection can make new evidence available without retraining the generator, but the application still needs to refresh affected records and indexes.

The [original RAG paper by Lewis et al.](https://arxiv.org/abs/2005.11401) combines a trained model with an external retrieval mechanism. Our educational system will use the broader engineering pattern of retrieval plus evidence-conditioned generation; it is not a reproduction of that paper's training procedure.

### Alternatives and their trade-offs

| Approach | When it helps | Main limitation for our project |
| --- | --- | --- |
| LLM without documents | General explanations and language tasks | Cannot establish what our private PDF says or where it says it |
| Put the entire PDF text in the prompt | A small document that fits comfortably | Repeated input cost, context limits, distracting text; still needs provenance |
| Search and show passages without generation | Users need direct evidence | Less synthesis; the user must assemble an answer |
| Fine-tune a model | Adapt behavior, style, or a recurring task | Training cost; weights do not provide a dependable page-level source map |
| RAG | A changing collection requiring evidence-backed answers | Adds parsing, search, context selection, and evaluation failures |

RAG does **not** require embeddings or a vector database. Keyword search, structured queries, or hybrid search can supply evidence. We will learn dense retrieval later because meaning-based matching is useful, then compare its weaknesses with alternatives.

RAG also does not guarantee truth or eliminate hallucinations. A document may be wrong, retrieval may miss the right passage, and the model may misuse evidence. A faithful summary of a false document can still be false in the world. We will measure these different failures separately.

## 3. Intended architecture: two flows

Document preparation happens when documents enter or change in the collection. Question answering happens for each question. Preparing reusable search representations avoids processing every PDF from scratch for every query.

```text
DOCUMENT PREPARATION - future phases
Local PDFs
   │
   ▼
Ingestion + document identity
   │
   ▼
Page-aware text extraction ──► source/page records
   │
   ▼
Chunking, retaining source/page relationships
   │
   ▼
Embeddings + searchable index ──► evidence store + metadata

QUESTION ANSWERING - future phases
Question ──► retrieval ──► selected evidence + evidence IDs
                                  │
                                  ▼
                    context augmentation + instructions
                                  │
                                  ▼
                         LLM answer with evidence IDs
                                  │
                                  ▼
             resolve IDs to source/pages + validate support
                                  │
                                  ▼
                     answer with traceable citations
```

The original PDF and its identity remain the source anchor. Metadata travels with the text rather than being reconstructed from a final answer. Our first stages will be deliberately simple; no distributed services are justified by three pages of teaching material.

**Phase 1 implements the workspace beneath this diagram, not the boxes in it.**

### Every stage has a different job

| Stage | Input → output | How it works / why we need it | Main trade-off and later validation |
| --- | --- | --- | --- |
| Document ingestion | A file → registered document and source identity | Accept and identify a source; retain filename/location and eventually a content fingerprint/version | Storage and duplicate/version handling; confirm intended file is registered |
| Text extraction | PDF pages → page text and metadata | Decode text/layout objects; image-only pages may require OCR | Speed versus reading-order/layout fidelity; compare text with the actual pages |
| Chunking | Page text → smaller evidence units and page mappings | Split text at chosen boundaries to fit retrieval/context needs | Small chunks can lose context; large chunks can mix topics; inspect boundaries and evidence coverage |
| Embedding generation | Text → numeric vector | A model maps text into a representation intended to preserve useful semantic relationships | Model quality, vector size, compute and memory; test known matching/nonmatching passages |
| Indexing | Text, vectors and metadata → searchable structures | Organize records for lookup; exact vector scans, approximate structures and lexical indexes are alternatives | Speed/memory/update complexity versus accuracy; check record counts and reference search results |
| Retrieval | Question → ranked evidence candidates | Search using the query's words or representation, then select candidates | More candidates can improve coverage but add noise and downstream cost; measure relevance and latency |
| Context augmentation | Question + selected evidence → model input | Package bounded evidence, IDs and instructions within a context budget | Include enough support without irrelevant material; inspect the actual input and token budget |
| Answer generation | Model input → answer text | Predict output tokens conditioned on the supplied evidence and instructions | Quality, latency and cost; compare claims with known answers and evidence |
| Citation attribution | Answer claims + evidence IDs → source/page references | Resolve IDs through trusted metadata; check whether cited evidence supports each claim | A valid location is easier to verify than semantic support; audit both separately |

Some implementations combine these jobs in one library call, but their failure modes remain distinct. We will first implement them explicitly so we can see intermediate outputs and diagnose mistakes.

Embeddings, indexing, ranking algorithms and token budgets are introduced here only by their roles. Their internal mathematics and experiments belong to the corresponding phases.

## 4. Design decisions for this phase

The current problem is to create a trustworthy place to learn and run experiments, without introducing the full application stack.

| Decision | Why it fits now | Alternative / when to revisit |
| --- | --- | --- |
| Python 3.14.6 | It is the interpreter already installed here | Use another supported Python if future native/ML dependencies require it; compatibility is not assumed |
| Built-in `venv` and pip | Isolate installed packages with no extra environment-manager dependency | uv or Conda can help with Python management or more complex dependency workflows |
| `requirements.in` plus pinned `requirements.txt` | Separate direct requirements from the observed resolved environment | A hash-verified lock workflow becomes valuable across platforms or deployments |
| JupyterLab | Read explanations, execute small cells, and inspect results together | A compatible editor can use the same notebook and kernel |
| Explicit ipykernel registration | Make the chosen Python environment visible as **Python (RAG learning)** | Re-register if the environment is recreated or moved |
| nbformat and nbclient | Check notebook structure and execute a fresh kernel automatically | Manual execution is useful for exploration but can hide out-of-order state |
| Notebook plus one verifier script | There is no reusable RAG application logic yet | Add `src/` and dedicated tests when reusable ingestion behavior arrives |

`venv` changes where Python looks for installed packages; it is not a separate operating system or security boundary. It uses the installed base interpreter. Shell activation only adjusts command lookup; `.venv/bin/python` selects the interpreter explicitly.

The four direct dependencies are JupyterLab (interface), ipykernel (Python execution backend), nbformat (notebook file/schema), and nbclient (execution driver). They also install their own dependencies. We are not installing a PDF parser, embedding framework, vector store, or LLM client in Phase 1.

A notebook has three distinct parts: the **file** stores Markdown/code/outputs; the **frontend** displays and edits it; the **kernel** is a live Python process holding variables. A saved output can outlive the process that created it. This is why we verify execution from a new kernel.

## 5. Implementation and execution: verify the environment

This first code cell imports only standard-library modules and Jupyter tooling. It locates the project by walking upward from the current working directory, so it works when launched from either the root or `notebooks/`. A missing project marker raises an explicit error instead of silently reading another directory.

`sys.executable` tells us which Python runs this cell. `sys.prefix` identifies its environment; `sys.base_prefix` identifies the base Python. We compare directories rather than resolving the executable symlink, because a virtual environment's executable may link to the base interpreter.

In [1]:
import json
import platform
import sys
from importlib.metadata import version
from pathlib import Path

import ipykernel
import jupyterlab
import nbclient
import nbformat

candidate_roots = [Path.cwd(), *Path.cwd().parents]
project_roots = [
    path for path in candidate_roots
    if (path / "requirements.in").is_file()
    and (path / "notebooks" / "01_rag_fundamentals.ipynb").is_file()
]
if not project_roots:
    raise FileNotFoundError("Launch this notebook inside the RAG project.")
PROJECT_ROOT = project_roots[0]

assert sys.prefix != sys.base_prefix, "Select the project virtual environment."
assert Path(sys.prefix).resolve() == (PROJECT_ROOT / ".venv").resolve(), (
    "Select Python (RAG learning), then restart the kernel."
)

environment_report = {
    "python": platform.python_version(),
    "recorded_python": (PROJECT_ROOT / ".python-version").read_text().strip(),
    "executable": Path(sys.executable).relative_to(PROJECT_ROOT).as_posix(),
    "platform": platform.system(),
    "architecture": platform.machine(),
    "in_virtual_environment": sys.prefix != sys.base_prefix,
    "packages": {
        name: version(name)
        for name in ("jupyterlab", "ipykernel", "nbformat", "nbclient")
    },
}
print(json.dumps(environment_report, indent=2))

{
  "python": "3.14.6",
  "recorded_python": "3.14.6",
  "executable": ".venv/bin/python",
  "platform": "Darwin",
  "architecture": "arm64",
  "in_virtual_environment": true,
  "packages": {
    "jupyterlab": "4.6.4",
    "ipykernel": "7.4.0",
    "nbformat": "5.11.1",
    "nbclient": "0.11.0"
  }
}


### Read the result before continuing

The executable should be inside this project's `.venv/bin/`, and `in_virtual_environment` should be `true`. We record the initial Python patch version; `.python-version` itself does not install or switch Python. A version mismatch is a reason to check compatibility and revalidate, rather than assuming a saved output applies.

Import success establishes that these packages can load in this interpreter. It does not prove that a notebook frontend can start or that future ML libraries support this Python. The automated verifier also starts a real kernel and checks all package pins.

In [2]:
expected_files = [
    "README.md",
    ".gitignore",
    ".python-version",
    "requirements.in",
    "requirements.txt",
    "scripts/verify_phase1.py",
    "notebooks/01_rag_fundamentals.ipynb",
    "data/sample/fixture_plan.json",
]
for relative_path in expected_files:
    assert (PROJECT_ROOT / relative_path).is_file(), relative_path
    print(f"Found: {relative_path}")
print("Project-file smoke check passed.")

Found: README.md
Found: .gitignore
Found: .python-version
Found: requirements.in
Found: requirements.txt
Found: scripts/verify_phase1.py
Found: notebooks/01_rag_fundamentals.ipynb
Found: data/sample/fixture_plan.json
Project-file smoke check passed.


### Why these files exist

The README is our setup guide, architectural decision record, and progress checkpoint. `requirements.in` expresses direct version constraints; `requirements.txt` records exact installed versions, including transitive packages and pip. `.gitignore` excludes the virtual environment, runtime state, and a designated private-document folder. No credentials are needed, so an `.env` file would serve no purpose yet.

The notebook is the lesson and experiment log. The verification script validates the notebook schema, starts a fresh project kernel, executes all cells, checks execution order/errors, and saves successful outputs. The fictional fixture plan provides controlled expectations for future PDF experiments. We do not create empty modules for future stages.

The pinned snapshot improves repeatability on the recorded platform. [`pip freeze`](https://pip.pypa.io/en/stable/cli/pip_freeze/) records installed packages; it is not a portable solver lock and does not hash downloaded artifacts. That limitation is explicit rather than hidden behind the word "reproducible."

## 6. Controlled sample: content planned for a future PDF

We need known facts to distinguish a system that works from one that merely sounds plausible. `fixture_plan.json` contains invented Cedar Labs content, three intended pages, and five manually authored questions. These are test expectations created by us, not answers discovered by a system.

**There is no PDF yet.** We prepare its content now and will create and parse it in Phase 2. This avoids adding PDF authoring or extraction dependencies just to complete the environment lesson. The planned page assignments are hypotheses to check against the actual generated PDF later.

Python's `json.loads` converts JSON text into dictionaries and lists. It does not extract PDF text. This cell reports the fixture status and contents without performing search.

In [3]:
fixture_path = PROJECT_ROOT / "data" / "sample" / "fixture_plan.json"
fixture_plan = json.loads(fixture_path.read_text(encoding="utf-8"))

print(fixture_plan["description"])
print("Status:", fixture_plan["status"])
print("Planned PDF:", fixture_plan["planned_source_location"])
for page in fixture_plan["pages"]:
    print(
        f"Planned physical page {page['physical_page_number']} "
        f"(printed label {page['printed_page_label']!r}): {page['title']}"
    )
print("Known questions:", len(fixture_plan["known_questions"]))

Entirely fictional teaching material. Page assignments below are authoring expectations, not extracted PDF evidence.
Status: planned_pdf_not_created_or_parsed
Planned PDF: data/sample/cedar_labs_handbook.pdf
Planned physical page 1 (printed label 'i'): Cedar Labs - fictional company overview
Planned physical page 2 (printed label '1'): Learning and workplace policy
Planned physical page 3 (printed label '2'): Product strategy and support
Known questions: 5


## 7. Provenance: preserve where a statement came from

**Provenance** is the record connecting information to its source and transformations. Text alone cannot tell us which file or page it came from. A document identifier distinguishes sources independently of their filenames; filenames can collide. A source location lets us find the original. In Phase 2 we will decide how file content, identity, and versions relate.

Our eventual records must retain:

| Information | Purpose |
| --- | --- |
| Document identifier, filename, source location | Locate the right source and distinguish similarly named files |
| Zero-based page index and one-based physical page number | Link library output to the reader's page position |
| Printed page label, when available | Keep publication numbering separate from physical numbering |
| Chunk identifier and chunk text, once chunking exists | Identify precisely which retrievable unit supplied evidence |
| Page relationships for each chunk | Retain all pages that contributed text, including spans and overlaps |

For a future multi-page chunk, a start-page field alone is insufficient. We will need per-page relationships, potentially with offsets into the chunk, to identify which part supports a claim. Overlap also means two chunk IDs can contain the same original words; repeated text is not independent corroboration.

Below we manually choose the second planned page with list position `1`. This is direct list access to a known position, not an algorithm finding relevant evidence.

In [4]:
manual_page_example = fixture_plan["pages"][1]
manual_provenance_example = {
    "status": "manually_authored_plan_not_extracted_evidence",
    "document_id": fixture_plan["document_id"],
    "planned_filename": fixture_plan["planned_filename"],
    "planned_source_location": fixture_plan["planned_source_location"],
    "page_index": manual_page_example["page_index"],
    "physical_page_number": manual_page_example["physical_page_number"],
    "printed_page_label": manual_page_example["printed_page_label"],
    "text": manual_page_example["text"],
}
print(json.dumps(manual_provenance_example, indent=2))

{
  "status": "manually_authored_plan_not_extracted_evidence",
  "document_id": "cedar-labs-handbook-v1-fixture",
  "planned_filename": "cedar_labs_handbook.pdf",
  "planned_source_location": "data/sample/cedar_labs_handbook.pdf",
  "page_index": 1,
  "physical_page_number": 2,
  "printed_page_label": "1",
  "text": "Each Cedar Labs employee has an annual learning budget of INR 24,000. Employees may work remotely up to three days per week. Learning purchases require approval from the employee's manager."
}


### Three meanings of "page"

1. **Page index:** a library's position, normally starting at zero. The second page has index `1`.
2. **Physical PDF page number:** the position in the PDF starting at one. That same page is physical page `2`. For our library convention, `physical_page_number = page_index + 1`.
3. **Printed page label:** a label such as `i`, `1`, or `A-3`. It may be printed in the content or stored as PDF page-label metadata. Those two sources of labels can themselves disagree. The second physical page in our plan is printed `1`.

Initially, our citations will use **physical PDF page numbers**, including covers and front matter. We will state that convention in the application. A reader using a PDF viewer's printed-label mode may need to switch to physical position to follow such a reference.

The following table is computed from the authored plan; it is not evidence that any PDF has these pages yet.

In [5]:
print(f"{'Page index':<14}{'Physical page':<17}Printed label")
for page in fixture_plan["pages"]:
    print(
        f"{page['page_index']:<14}"
        f"{page['physical_page_number']:<17}"
        f"{page['printed_page_label']}"
    )

assert manual_page_example["page_index"] == 1
assert manual_page_example["physical_page_number"] == 2
assert manual_page_example["printed_page_label"] == "1"

Page index    Physical page    Printed label
0             1                i
1             2                1
2             3                2


## 8. Experiments: inspect assumptions before automating them

### Experiment A - change the page-number convention

**Prediction:** if we display a zero-based index as a citation page, the second page will be mislabeled as page `1`. If we display a printed label as a physical page, the same error can look reasonable because both are valid numbers.

The next cell enumerates these choices for one known page. The only correct choice for our declared convention is its one-based physical position. This experiment catches a bookkeeping error, not citation support for a generated answer.

In [6]:
page_number_choices = {
    "mistakenly use library index": manual_page_example["page_index"],
    "mistakenly use printed label": manual_page_example["printed_page_label"],
    "use physical position": manual_page_example["physical_page_number"],
}
for convention, displayed_page in page_number_choices.items():
    print(f"{convention}: {displayed_page}")

assert manual_page_example["page_index"] + 1 == manual_page_example["physical_page_number"]
assert str(manual_page_example["physical_page_number"]) != manual_page_example["printed_page_label"]

mistakenly use library index: 1
mistakenly use printed label: 1
use physical position: 2


### Experiment B - remove provenance

**Prediction:** the wording remains readable when metadata is discarded, but the resulting string cannot identify its source page. Python's string type holds characters; it has no built-in concept of a PDF, source document, or citation.

This demonstrates why provenance must be preserved throughout the pipeline. Attempting to infer a page number from the answer afterward would be unreliable.

In [7]:
text_without_provenance = manual_provenance_example["text"]
print("Text-only type:", type(text_without_provenance).__name__)
print("Text-only value:", text_without_provenance)
print("Metadata retained in the annotated record:")
print([key for key in manual_provenance_example if key != "text"])
assert isinstance(text_without_provenance, str)

Text-only type: str
Text-only value: Each Cedar Labs employee has an annual learning budget of INR 24,000. Employees may work remotely up to three days per week. Learning purchases require approval from the employee's manager.
Metadata retained in the annotated record:
['status', 'document_id', 'planned_filename', 'planned_source_location', 'page_index', 'physical_page_number', 'printed_page_label']


### Experiment C - manually vary the available evidence

Consider the known question: **"What is the annual learning budget per employee?"** The planned second page explicitly states **INR 24,000**. The first page discusses headquarters and founding year instead.

| Evidence supplied by hand | Can a reader establish the budget from this evidence? | Responsible document-based response |
| --- | --- | --- |
| Planned page 1 only | No | State that the supplied evidence does not establish the budget |
| Planned page 2 only | Yes | State INR 24,000 and cite physical page 2, once verified in the actual PDF |
| Planned pages 1 and 2 | Yes | Use the budget statement; page 1 is unnecessary for this claim |
| All three pages, asked about 2025 revenue | No | Acknowledge that the document does not supply the requested revenue |

These outcomes are **human-authored judgments**, not model outputs. More context is not automatically better: it can add irrelevant material and consumes context budget. A future experiment will measure the effect on actual retrieval and generation.

The code below only displays two manually chosen pages for you to inspect. It does not rank passages, answer a question, or check semantic support automatically.

In [8]:
print("MANUAL INSPECTION: compare the evidence yourself.")
print()
for known_position in (0, 1):
    chosen_page = fixture_plan["pages"][known_position]
    print(f"Planned physical page {chosen_page['physical_page_number']}:")
    print(chosen_page["text"])
    print()

known_budget_case = fixture_plan["known_questions"][1]
print("Human-authored expected answer:", known_budget_case["expected_answer"])
print("Human-authored expected physical pages:", known_budget_case["expected_physical_pages"])

MANUAL INSPECTION: compare the evidence yourself.

Planned physical page 1:
Cedar Labs is a fictional company created for RAG learning exercises. Its headquarters is in Pune. The company was founded in 2022. This handbook contains invented facts and is not real company policy.

Planned physical page 2:
Each Cedar Labs employee has an annual learning budget of INR 24,000. Employees may work remotely up to three days per week. Learning purchases require approval from the employee's manager.

Human-authored expected answer: INR 24,000
Human-authored expected physical pages: [2]


## 9. Citation attribution is a separate engineering problem

Our eventual example answer could be: "The annual learning budget is INR 24,000 per employee [Source: cedar_labs_handbook.pdf, physical page 2]." **Today this is an illustrative target based on the plan, not a verified PDF citation or a generated answer.**

The future mechanism will give selected evidence stable IDs, ask the generator to reference only those IDs, and resolve IDs to filename/page metadata in application code. Unknown IDs must be rejected. The application should derive source names and page numbers from its registry rather than trusting invented source strings in generated text.

This mechanism establishes traceability, not semantic truth. We must also check whether the passage supports the associated claim and whether every document-derived claim has adequate support. Even a real evidence ID can be attached to the wrong sentence. If evidence is missing, the desired response is to acknowledge the limitation, not invent a fact or citation.

| Separate question | Example failure | How we will evaluate it later |
| --- | --- | --- |
| Was the document ingested? | A file never entered the collection | Compare expected and registered sources |
| Was relevant evidence retrieved? | The handbook is present but the budget page was missed | Known relevant passages/pages and retrieval metrics |
| Was an answer produced? | The model returns an empty response | Output/error checks; existence alone says little about quality |
| Is the citation location valid? | A nonexistent source or out-of-range page | Resolve evidence IDs and check source/page records |
| Does the cited evidence support the claim? | Page 1 is cited for the budget | Claim-to-passage review and controlled reference cases |
| Is the answer correct and complete? | Wrong currency or a missing part of a two-part question | Compare with reference answers and inspect evidence |

We start with controlled expectations rather than an LLM judge. More advanced evaluation belongs to later phases, and human inspection remains useful.

## 10. Validation: what we can honestly check now

We can check the environment, notebook execution, and internal consistency of our authored fixture. The following assertions verify that planned pages are consecutive, that index-to-position conversion holds, and that expected page references stay within the plan. The exact supporting-text check catches a typo in an authored expectation. It is not a general semantic entailment test.

Python raises `AssertionError` if a check fails. Assertions are appropriate for these learning checks; production input validation should use explicit exceptions because Python can disable assertions with optimization flags.

Run this notebook without `python -O`. Our terminal verifier also checks the notebook schema, all saved execution counts and errors, kernel interpreter, and exact installed package pins.

In [9]:
pages = fixture_plan["pages"]
expected_physical_pages = set(range(1, len(pages) + 1))

assert fixture_plan["status"] == "planned_pdf_not_created_or_parsed"
assert len(pages) == 3
assert [page["page_index"] for page in pages] == list(range(len(pages)))
assert [page["physical_page_number"] for page in pages] == list(range(1, len(pages) + 1))
assert all(page["physical_page_number"] == page["page_index"] + 1 for page in pages)
assert all(page["text"].strip() for page in pages)
assert len({case["id"] for case in fixture_plan["known_questions"]}) == 5

for case in fixture_plan["known_questions"]:
    assert set(case["expected_physical_pages"]).issubset(expected_physical_pages)
    if case["supporting_text"] is not None:
        assert len(case["expected_physical_pages"]) == 1
        authored_page = pages[case["expected_physical_pages"][0] - 1]
        assert case["supporting_text"] in authored_page["text"]

unknown_case = fixture_plan["known_questions"][-1]
assert unknown_case["expected_answer"] is None
assert unknown_case["expected_physical_pages"] == []
print("Authored fixture consistency checks passed: 3 planned pages, 5 known questions.")
print("No PDF extraction, search, generated-answer, or citation-support evaluation was run.")

Authored fixture consistency checks passed: 3 planned pages, 5 known questions.
No PDF extraction, search, generated-answer, or citation-support evaluation was run.


### A small failure demonstration

An off-by-one error should fail loudly. Here we intentionally assert the wrong convention, catch only the expected `AssertionError`, and print why it failed. The notebook continues because this is a controlled example. A real unexpected failure should not be swallowed.

In [10]:
try:
    assert manual_page_example["page_index"] == manual_page_example["physical_page_number"], (
        "Zero-based index and one-based physical page must differ by one."
    )
except AssertionError as expected_error:
    print("Expected failure caught:", expected_error)
else:
    raise RuntimeError("The deliberate page-number mistake unexpectedly passed.")

Expected failure caught: Zero-based index and one-based physical page must differ by one.


## 11. Common mistakes and phase-appropriate edge cases

| Mistake / case | Why it matters | What we do now / when we address it |
| --- | --- | --- |
| Install into one Python, run another kernel | Imports and versions differ | Use `.venv/bin/python` and inspect `sys.executable` |
| Execute cells out of order | Variables survive even when earlier code changes | Restart and run all; verify with a fresh kernel |
| Trust saved outputs | They may come from an old environment | Rerun and check current package versions |
| Treat authored JSON as extracted PDF evidence | It hides parsing failures | Label plans explicitly; compare with real pages in Phase 2 |
| Treat a filename as document identity | Different files can share a name | Preserve identifiers and source locations; decide versioning during ingestion |
| Use printed labels as physical pages | Covers and front matter shift numbering | Keep numbering concepts separate from the beginning |
| Drop page mappings when chunking later | Source attribution becomes ambiguous | Preserve relationships; design chunk spans in Phase 3 |
| Assume any retrieved passage answers the question | Presence is not relevance or support | Keep retrieval and answer/citation evaluation distinct |
| Empty or scanned PDF pages | Empty extracted text may mean an image, not an empty page | Inspect and flag in Phase 2; do not solve OCR here |
| A document contains instructions | Retrieved text can be untrusted | Treat content as evidence, not authority; implement protections when generation begins |

No API key, external model, or paid service is required. Initial dependency installation uses a package index; the installed lesson makes no external service calls.

## 12. Key takeaways and checkpoint

- RAG adds external evidence selection to generation. It is not a guarantee of correctness and need not use vectors.
- Document preparation and question answering are separate flows. Each stage has inputs, outputs, and its own validation needs.
- Provenance belongs with the evidence from the start. Physical page numbers, indexes, and printed labels are different concepts.
- A source/page reference can be valid without supporting the claim. Traceability and semantic support need separate checks.
- Virtual environments isolate packages; kernels execute cells. A fresh execution is stronger evidence than a saved output.
- Our executable work is limited to environment verification and authored-data inspection. We have prepared controlled facts, not built a RAG pipeline.

**Self-check:** explain why page index `1`, physical page `2`, and printed label `1` can describe the same page. Then explain why a citation to a real page could still be wrong. If either is unclear, pause and ask before we add more code.

**Known limitations:** the sample PDF does not exist yet; no PDF parser, chunker, embedding model, index, retriever, or generator has been implemented. Future dependency compatibility, retrieval quality, and answer quality are untested. The environment snapshot lacks artifact hashes and is specific to the recorded platform.

**Proposed Phase 2:** create the fictional three-page PDF, inspect its actual pages, extract text page by page, preserve document/page metadata, compare extraction with known facts, and identify empty or image-only-page limitations. We will discuss a parser and its alternatives when selecting that dependency.

**STOP after Phase 1. Phase 2 requires your explicit approval.** Conceptual questions do not authorize the next implementation phase.

## References and local workflow

These primary sources support the concepts and setup choices; no web access is needed to execute this notebook.

- [Lewis et al., Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks](https://arxiv.org/abs/2005.11401) - foundational retrieval-plus-generation research.
- [Python: virtual environments](https://docs.python.org/3.14/library/venv.html) - environment behavior and interpreter relationships.
- [Project Jupyter: installation](https://jupyter.org/install) - installing and launching the notebook interface.
- [IPython: kernel installation](https://ipython.readthedocs.io/en/stable/install/kernel_install.html) - environment-specific kernels.
- [pip: freeze](https://pip.pypa.io/en/stable/cli/pip_freeze/) - installed-package snapshots and their limits.
- [nbclient: notebook execution](https://nbclient.readthedocs.io/en/latest/client.html) - fresh-kernel execution and saved outputs.

From the project root, run `.venv/bin/python scripts/verify_phase1.py` to repeat automated validation. See `README.md` for exact setup and JupyterLab launch commands. The verifier saves outputs only after a successful run. Keep this notebook and the README together when resuming the project.